# House Prices - Fase 4: XGBoostXGBoost e um dos modelos mais poderosos para dados tabulares.**O que vamos aprender:**1. O que e XGBoost (Gradient Boosting)2. Como ele difere do Random Forest3. Hiperparametros principais4. Comparar com Ridge e Random Forest5. Gerar submissao final

## O que e XGBoost?XGBoost = **X**treme **G**radient **Boost**ing### Random Forest vs XGBoost| Caracteristica | Random Forest | XGBoost ||---------------|---------------|---------|| Tipo | Bagging (arvores paralelas) | Boosting (arvores sequenciais) || Como funciona | Cada arvore independente | Cada arvore corrige erros da anterior || Overfitting | Menos propenso | Mais propenso (precisa de tuning) || Velocidade | Mais rapido | Mais lento || Performance | Muito bom | Geralmente melhor |### Analogia: Random Forest100 especialistas independentes votam e faz-se a media.### Analogia: XGBoostUm especialista erra -> outro tenta corrigir o erro -> outro tenta corrigir o resto...E como uma equipe que vai melhorando a cada rodada.### Como o Boosting funciona:1. Faz uma previsao simples2. Calcula o erro (residuo)3. Treina um novo modelo para prever o erro4. Soma a nova previsao a anterior5. Repete N vezesPrevisao final = previsao1 + previsao2 + previsao3 + ... + previsaoN

In [ ]:
%matplotlib inlineimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport osfrom sklearn.linear_model import Ridgefrom sklearn.ensemble import RandomForestRegressorfrom sklearn.model_selection import cross_val_score, KFoldfrom sklearn.metrics import mean_squared_errorimport xgboost as xgbsns.set_style('whitegrid')_cwd = os.getcwd()if _cwd.endswith('notebooks'):    BASE_DIR = os.path.dirname(_cwd)else:    BASE_DIR = _cwdDATA_DIR = os.path.join(BASE_DIR, 'data')REPORTS_DIR = os.path.join(BASE_DIR, 'reports')RANDOM_SEED = 42print('XGBoost version:', xgb.__version__)print('RANDOM_SEED:', RANDOM_SEED)

In [ ]:
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train_clean.csv'))y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train_clean.csv')).squeeze()X_test = pd.read_csv(os.path.join(DATA_DIR, 'X_test_clean.csv'))test_original = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))def rmsle(y_true, y_pred):    y_pred = np.maximum(y_pred, 0)    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))def evaluate_rmse(model, X, y, cv=5):    kf = KFold(n_splits=cv, shuffle=True, random_state=RANDOM_SEED)    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')    return np.sqrt(-scores)print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Modelo 1: XGBoost BasicoVamos comecar com um XGBoost simples e depois melhorar.### Hiperparametros principais:| Parametro | O que faz | Valor tipico ||-----------|-----------|-------------|| n_estimators | Numero de arvores (iteracoes) | 100-1000 || max_depth | Profundidade de cada arvore | 3-10 || learning_rate | Tamanho do passo (quanto cada arvore corrige) | 0.01-0.3 || subsample | Fracao de dados usada por arvore | 0.7-1.0 || colsample_bytree | Fracao de features usada por arvore | 0.7-1.0 || reg_alpha | Regularizacao L1 | 0-10 || reg_lambda | Regularizacao L2 | 0-10 |**learning_rate e o mais importante:**- Muito alto: converge rapido mas pode overfitting- Muito baixo: precisa de mais arvores, mais lento- Regra: learning_rate baixo + n_estimators alto = melhor

In [ ]:
print('=' * 60)print('XGBoost - Modelo Basico'print('=' * 60)xgb_model = xgb.XGBRegressor(    n_estimators=200,    max_depth=6,    learning_rate=0.1,    subsample=0.8,    colsample_bytree=0.8,    random_state=RANDOM_SEED,    n_jobs=-1,    verbosity=0)xgb_scores = evaluate_rmse(xgb_model, X_train, y_train)xgb_model.fit(X_train, y_train)y_pred_xgb = xgb_model.predict(X_train)xgb_rmsle = rmsle(y_train, y_pred_xgb)print(f'XGBoost (basico): RMSE={xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})')print(f'                  RMSLE={xgb_rmsle:.4f}')

## Modelo 2: XGBoost OtimizadoAgora vamos testar com hiperparametros mais agressivos.Tecnica: learning_rate baixo + n_estimators alto = melhor generalizacaoE como estudar para uma prova: passos pequenos e muitos revisam melhor que passos grandes e poucos.

In [ ]:
print('=' * 60)print('XGBoost - Modelo Otimizado'print('=' * 60)xgb_opt = xgb.XGBRegressor(    n_estimators=500,    max_depth=4,    learning_rate=0.05,    subsample=0.7,    colsample_bytree=0.7,    reg_alpha=0.1,    reg_lambda=1.0,    random_state=RANDOM_SEED,    n_jobs=-1,    verbosity=0)xgb_opt_scores = evaluate_rmse(xgb_opt, X_train, y_train)xgb_opt.fit(X_train, y_train)y_pred_xgb_opt = xgb_opt.predict(X_train)xgb_opt_rmsle = rmsle(y_train, y_pred_xgb_opt)print(f'XGBoost (otimizado): RMSE={xgb_opt_scores.mean():.4f} (+/- {xgb_opt_scores.std():.4f})')print(f'                     RMSLE={xgb_opt_rmsle:.4f}')

## Comparacao FinalVamos comparar todos os modelos lado a lado.

In [ ]:
print('=' * 60)print('COMPARACAO FINAL DE MODELOS'print('=' * 60)# Ridgeridge = Ridge(alpha=10.0, random_state=RANDOM_SEED)ridge.fit(X_train, y_train)ridge_pred = ridge.predict(X_train)ridge_rmsle = rmsle(y_train, ridge_pred)kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)ridge_cv = np.sqrt(-cross_val_score(ridge, X_train, y_train, cv=kf, scoring='neg_mean_squared_error'))# Random Forestrf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=RANDOM_SEED, n_jobs=-1)rf.fit(X_train, y_train)rf_pred = rf.predict(X_train)rf_rmsle = rmsle(y_train, rf_pred)rf_cv = np.sqrt(-cross_val_score(rf, X_train, y_train, cv=kf, scoring='neg_mean_squared_error'))# XGBoostxgb_pred = xgb_opt.predict(X_train)xgb_rmsle = rmsle(y_train, xgb_pred)# Tabelaprint(f'\n{"Modelo":<25} {"RMSE (CV)":<15} {"RMSLE":<10}')print('-' * 50)print(f'{"Ridge":<25} {ridge_cv.mean():<15.4f} {ridge_rmsle:<10.4f}')print(f'{"Random Forest":<25} {rf_cv.mean():<15.4f} {rf_rmsle:<10.4f}')print(f'{"XGBoost (otimizado)":<25} {xgb_opt_scores.mean():<15.4f} {xgb_rmsle:<10.4f}')# Graficofig, axes = plt.subplots(1, 2, figsize=(14, 6))models = ['Ridge', 'Random Forest', 'XGBoost']rmse_vals = [ridge_cv.mean(), rf_cv.mean(), xgb_opt_scores.mean()]rmsle_vals = [ridge_rmsle, rf_rmsle, xgb_rmsle]colors = ['steelblue', 'coral', 'forestgreen']axes[0].bar(models, rmse_vals, color=colors, edgecolor='white', alpha=0.8)axes[0].set_title('RMSE Medio (CV)', fontweight='bold')axes[0].set_ylabel('RMSE (log scale)')for i, v in enumerate(rmse_vals):    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')axes[0].grid(True, alpha=0.3)axes[1].bar(models, rmsle_vals, color=colors, edgecolor='white', alpha=0.8)axes[1].set_title('RMSLE (Kaggle metric)', fontweight='bold')axes[1].set_ylabel('RMSLE')for i, v in enumerate(rmsle_vals):    axes[1].text(i, v + 0.0003, f'{v:.4f}', ha='center', fontweight='bold')axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.savefig('../reports/f04_model_comparison.png', dpi=150, bbox_inches='tight')print('\nGrafico salvo: reports/f04_model_comparison.png')

## Feature Importance do XGBoostXGBoost tambem mostra quais features sao mais importantes.Vamos comparar com o Random Forest.

In [ ]:
importances_xgb = pd.Series(xgb_opt.feature_importances_, index=X_train.columns)top10_xgb = importances_xgb.sort_values(ascending=False).head(10)importances_rf = pd.Series(rf.feature_importances_, index=X_train.columns)top10_rf = importances_rf.sort_values(ascending=False).head(10)fig, axes = plt.subplots(1, 2, figsize=(16, 7))top10_rf.sort_values().plot(kind='barh', ax=axes[0], color='coral', edgecolor='white')axes[0].set_title('Random Forest - Top 10 Features', fontweight='bold')axes[0].set_xlabel('Importancia')top10_xgb.sort_values().plot(kind='barh', ax=axes[1], color='forestgreen', edgecolor='white')axes[1].set_title('XGBoost - Top 10 Features', fontweight='bold')axes[1].set_xlabel('Importancia')plt.tight_layout()plt.savefig('../reports/f04_feature_importance.png', dpi=150, bbox_inches='tight')print('Top 10 - Random Forest:')for i, (feat, imp) in enumerate(top10_rf.items(), 1):    print(f'  {i:2d}. {feat:<25} {imp:.4f}')print('\nTop 10 - XGBoost:')for i, (feat, imp) in enumerate(top10_xgb.items(), 1):    print(f'  {i:2d}. {feat:<25} {imp:.4f}')

## Gerar Submissao FinalUsando o melhor modelo (XGBoost) para gerar a submissao.

In [ ]:
print('GERANDO SUBMISSION - XGBoost')y_pred_log = xgb_opt.predict(X_test)y_pred_real = np.expm1(y_pred_log)submission = pd.DataFrame({'Id': test_original['Id'], 'SalePrice': y_pred_real})submission.to_csv('../data/submission_xgb.csv', index=False)print(f'Submission: {len(submission)} previsoes')print(f'Media: R$ {submission["SalePrice"].mean():,.0f}')print(f'Min: R$ {submission["SalePrice"].min():,.0f}')print(f'Max: R$ {submission["SalePrice"].max():,.0f}')print(submission.head())print('\nPara submeter ao Kaggle:')print('kaggle competitions submit -c house-prices-advanced-regression-techniques -f data/submission_xgb.csv -m "XGBoost optimized"')

## Resumo da Fase 4**Modelos testados:**- Ridge (alpha=10): RMSE=0.1130, RMSLE=0.0078- Random Forest (100): RMSE=0.1357, RMSLE=0.0040- XGBoost (basico): ver resultado acima- XGBoost (otimizado): ver resultado acima**Proximos passos:**- Tuning fino com GridSearchCV- Ensemble (media de varios modelos)- Fase 5: Documentacao